# JSMA Interactive Challenge

This challenge notebook applies the course's targeted Jacobian-based Saliency Map Attack to the authenticated HTB instance. It keeps pixels in the server's `[0, 1]` space, computes gradients locally from downloaded weights, and verifies the quantized PNG before any optional submission.

## 1. Imports and target configuration

The target is the supplied HTB instance. `SUBMIT_TO_SERVER` remains false by default: `/predict` is a safe verification request, while `/submit` is the action that may consume the challenge attempt.

In [ ]:
import base64
import io
from pathlib import Path

import numpy as np
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

BASE_URL = 'http://154.57.164.67:32536'
REQUEST_TIMEOUT = 20
OUTPUT_DIR = Path('output/jsma_challenge')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_PATH = OUTPUT_DIR / 'jsma_weights.pth'
CANDIDATE_PATH = OUTPUT_DIR / 'jsma_candidate.png'
SUBMIT_TO_SERVER = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(1337)
print('Using device:', DEVICE)

## 2. Image transport and challenge fetch

The API transports a grayscale PNG as base64 text. The decode formula is `x = pixel / 255`; read aloud, ‘x equals pixel divided by 255.’ This puts the image in the same `[0,1]` coordinate system used by the attack.

In [ ]:
def x01_from_b64_png(encoded):
    image = Image.open(io.BytesIO(base64.b64decode(encoded))).convert('L')
    return np.asarray(image, dtype=np.float32) / 255.0

def b64_from_x01(x01):
    pixels = np.rint(np.clip(x01, 0.0, 1.0) * 255).astype(np.uint8)
    buffer = io.BytesIO()
    Image.fromarray(pixels, mode='L').save(buffer, format='PNG')
    return base64.b64encode(buffer.getvalue()).decode('ascii')

health = requests.get(f'{BASE_URL}/health', timeout=REQUEST_TIMEOUT); health.raise_for_status()
challenge_response = requests.get(f'{BASE_URL}/challenge', timeout=REQUEST_TIMEOUT); challenge_response.raise_for_status()
challenge = challenge_response.json()
original_x01 = x01_from_b64_png(challenge['image_b64'])
true_label = int(challenge['label'])
print({'shape': original_x01.shape, 'true_label': true_label, 'challenge_keys': sorted(challenge)})

## 3. Reproduce the server classifier

A Jacobian is a table of partial derivatives. For class `k` and pixel `i`, `J[k,i]` means ‘how much class-k's logit changes when pixel-i changes.’ We download the server weights so PyTorch can calculate those derivatives locally. Normalization stays inside the wrapper, while the attack still edits legal `[0,1]` pixels.

In [ ]:
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

class SimpleClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25); self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128); self.fc2 = nn.Linear(128, num_classes)
    def forward(self, x):
        x = F.relu(self.conv1(x)); x = F.relu(self.conv2(x)); x = F.max_pool2d(x, 2)
        x = self.dropout1(x); x = torch.flatten(x, 1); x = F.relu(self.fc1(x)); x = self.dropout2(x)
        return self.fc2(x)

class ChallengeModel(nn.Module):
    def __init__(self, classifier):
        super().__init__(); self.classifier = classifier
    def forward(self, x01):
        return self.classifier((x01 - MNIST_MEAN) / MNIST_STD)

weights_response = requests.get(f'{BASE_URL}/weights', timeout=REQUEST_TIMEOUT); weights_response.raise_for_status()
WEIGHTS_PATH.write_bytes(weights_response.content)
classifier = SimpleClassifier().to(DEVICE)
classifier.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE)); classifier.eval()
model = ChallengeModel(classifier).to(DEVICE).eval()
original = torch.from_numpy(original_x01).unsqueeze(0).unsqueeze(0).to(DEVICE)
with torch.no_grad(): local_pred = int(model(original).argmax(1).item())
print({'local_prediction': local_pred, 'true_label': true_label})
assert local_pred == true_label, 'Stop: local model does not reproduce the clean label.'

## 4. Jacobian and pairwise saliency

For target class `t`, `alpha` is the gradient of the target logit and `beta` is the sum of competitor gradients. For a pair `(p,q)`, we use `alpha_pq = alpha_p + alpha_q` and `beta_pq = beta_p + beta_q`; read aloud, ‘alpha sub p-q equals alpha sub p plus alpha sub q.’ The saliency score is `alpha_pq * abs(beta_pq)`, subject to increasing-target/decreasing-competitor sign constraints.

In [ ]:
def jacobian_logits(x, model, classes=10):
    rows = []
    for class_id in range(classes):
        model.zero_grad(set_to_none=True)
        x_class = x.detach().clone().requires_grad_(True)
        model(x_class)[0, class_id].backward()
        rows.append(x_class.grad.detach().flatten().cpu().numpy())
    return np.stack(rows)

def pair_saliency(alpha, beta, available, direction='increase', top_k=128):
    candidates = np.flatnonzero(available)
    if top_k is not None and candidates.size > top_k:
        base = alpha if direction == 'increase' else -alpha
        candidates = candidates[np.argsort(base[candidates])[-top_k:]]
    best = (-1, -1, 0.0)
    for a in range(len(candidates)):
        for b in range(a + 1, len(candidates)):
            p, q = int(candidates[a]), int(candidates[b])
            a_pq, b_pq = alpha[p] + alpha[q], beta[p] + beta[q]
            valid = (a_pq > 0 and b_pq < 0) if direction == 'increase' else (a_pq < 0 and b_pq > 0)
            score = float(a_pq * abs(b_pq)) if valid else 0.0
            if score > best[2]: best = (p, q, score)
    return best

def apply_pair(x, p, q, theta=1.0, increase=True):
    flat = x.detach().clone().flatten()
    step = theta if increase else -theta
    flat[p] = torch.clamp(flat[p] + step, 0.0, 1.0); flat[q] = torch.clamp(flat[q] + step, 0.0, 1.0)
    return flat.reshape_as(x)

## 5. Targeted pairwise JSMA attack

The attack is targeted: it tries to make a chosen target class win. Each iteration computes a fresh Jacobian, evaluates both directions, selects the larger valid score, changes two pixels, and masks them out. `gamma` is a fraction of the 784 MNIST pixels; `floor(gamma × 784 / 2)` is the maximum number of pair iterations.

In [ ]:
TARGET_LABEL = (true_label + 1) % 10  # Change if the challenge specifies a different target.
THETA, GAMMA, MAX_ITER, TOP_K = 1.0, 0.15, 90, 128
MAX_PIXELS = int(GAMMA * int(np.prod(original.shape[1:])))

def run_jsma(original, target):
    x_adv = original.detach().clone(); available = np.ones(x_adv.numel(), dtype=bool); changed = 0
    for iteration in range(MAX_ITER):
        with torch.no_grad():
            if int(model(x_adv).argmax(1).item()) == target: return x_adv, changed, iteration + 1
        if changed + 2 > MAX_PIXELS: break
        jac = jacobian_logits(x_adv, model)
        alpha = jac[target]; beta = jac.sum(axis=0) - alpha
        inc = pair_saliency(alpha, beta, available, 'increase', TOP_K)
        dec = pair_saliency(alpha, beta, available, 'decrease', TOP_K)
        p, q, score = inc if inc[2] >= dec[2] else dec
        increase = inc[2] >= dec[2]
        if score <= 0 or p < 0 or q < 0: break
        x_adv = apply_pair(x_adv, p, q, THETA, increase); available[p] = available[q] = False; changed += 2
    return x_adv, changed, iteration + 1

adversarial, pixels_changed, iterations = run_jsma(original, TARGET_LABEL)
with torch.no_grad(): predicted = int(model(adversarial).argmax(1).item())
print({'target': TARGET_LABEL, 'prediction': predicted, 'success': predicted == TARGET_LABEL, 'pixels_changed': pixels_changed, 'iterations': iterations})

## 6. PNG round-trip and server verification

The optimizer uses floating-point tensors, but the challenge receives 8-bit PNG values. Quantization can change a borderline result, so we encode and decode first, then verify the actual bytes with `/predict`. This is a best-practice boundary check, not an optional cosmetic step.

In [ ]:
candidate_b64 = b64_from_x01(adversarial[0, 0].detach().cpu().numpy())
candidate_x01 = x01_from_b64_png(candidate_b64)
Image.fromarray(np.rint(candidate_x01 * 255).astype(np.uint8), mode='L').save(CANDIDATE_PATH)
candidate_tensor = torch.from_numpy(candidate_x01).unsqueeze(0).unsqueeze(0).to(DEVICE)
with torch.no_grad(): candidate_pred = int(model(candidate_tensor).argmax(1).item())
print({'candidate_prediction': candidate_pred, 'target': TARGET_LABEL, 'saved_candidate': str(CANDIDATE_PATH.resolve())})
assert candidate_pred == TARGET_LABEL, 'PNG quantization lost the targeted prediction.'
predict = requests.post(f'{BASE_URL}/predict', json={'image_b64': candidate_b64}, timeout=REQUEST_TIMEOUT); predict.raise_for_status()
print('Server prediction:', predict.json())

## 7. Optional submission

Submission is deliberately isolated. Review the saved PNG, target prediction, and server response before changing the switch to `True`. The notebook never submits automatically.

In [ ]:
if SUBMIT_TO_SERVER:
    submit = requests.post(f'{BASE_URL}/submit', json={'image_b64': candidate_b64}, timeout=REQUEST_TIMEOUT)
    print('HTTP status:', submit.status_code); print('Server response:', submit.json()); submit.raise_for_status()
else:
    print('Submission skipped. Set SUBMIT_TO_SERVER=True only after reviewing the verified candidate.')